# 2026 DEFRA, local council and Urban Observatory sensor, data analysis

Following a meeting with council member (NAME), we recognise that elements of the Urban Observatory (UO) data are flawed. We begin by looking at a thorough data analysis of the UO MESH and MONITOR sensors against the offical, calibrated DEFRA approved and owned AURN instrumentation and locally-managed automatic monitoring sensors. Any and all analysis has been conducted on 2026 data, specifically March.

## Summary

We have highlighted the following as causes for concern and urgent attention:
- UO MESH sensors do not have any validity and need immediate replacement before being relied upon, this goes for any ongoing work that is using these sensors.
- UO MESH and MONITOR sensors have faulty wind direction and speed data, shown as constant over the course of a day, week and month.
- DEFRA Newcastle Centre is rounding all values to nearest integer
- To validate the monitor sensors we compared them to their closest approved DEFRA sensors, most proved good agreement if not some nonlinear relationship however three were a cause for concern. Some of the disagreement can be attributed to distance between sensors however one sensor proved there to be other factors at play. Some advice on what this might be would be good in order to keep validation of the MONITOR sensors.


## Import Libraries

We begin by importing all required Python libraries:
- `numpy`, `pandas`, `geopandas`: data handling
- `gtda`, `statsmodels`, `geopy`: analysis
- `matplotlib`, `seaborn`, `contextily`: visualization
- `gpflow`, `tensorflow`: Gaussian Process modeling
- `uo_pyfetch`: Urban Observatory data library
- Other utilities for preprocessing and evaluation

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import contextily as ctx
import matplotlib.pyplot as plt
import seaborn as sns
import uo_pyfetch
import datetime
from IPython.display import HTML
from gtda.time_series import SingleTakensEmbedding
import plotly.graph_objects as go
from gtda.plotting import plot_point_cloud
from gtda.homology import VietorisRipsPersistence
from geopy.distance import geodesic
from statsmodels.tsa.stattools import acf
import math
from pathlib import Path

def to_float64(X):
    """
    Convert input data to float64.

    GPflow requires input data to be in float64 format.
    This function ensures compatibility and prevents dtype errors.
    """
    return np.asarray(X, dtype=np.float64)

In [ ]:
# test

PM25_2025_Jan = uo_pyfetch.get_sensor_data(
            start= datetime.datetime(2018, 1, 1),
            end= datetime.datetime(2018, 1, 31),
            variables=["PM2.5"],
            limit=-1
        )

In [ ]:
dfs = []

for month in range(1, 6):
    start = datetime.datetime(2026, month, 1)

    if month == 5:
        end = datetime.datetime(2026, month, 13, 23, 59, 59)
    else:
        end = datetime.datetime(2026, month + 1, 1) - datetime.timedelta(seconds=1)

    print(f"Fetching {start:%Y-%m}")

    try:
        df = uo_pyfetch.get_sensor_data(
            start=start,
            end=end,
            variables=["PM2.5"],
            limit=-1
        )
        dfs.append(df)

    except Exception as e:
        print(f"Failed for {month}: {e}")

PM25_2026 = pd.concat(dfs, ignore_index=True)

PM25_2026.to_csv('2026uptoMay27-PM25-UO.csv')

In [ ]:
PM25_2026 = pd.read_csv("2026uptoMay27-PM25-UO.csv")

PM25_2026["Timestamp"] = pd.to_datetime(PM25_2026["Timestamp"], errors="coerce")

uo_march_2026 = PM25_2026[
    (PM25_2026["Timestamp"] >= "2026-03-01") &
    (PM25_2026["Timestamp"] < "2026-03-31")
]


In [ ]:
"""
Filter and resample PM2.5 data for a specific sensor and year.

Steps:
1. Convert timestamp to datetime format
2. Filter dataset by sensor name
3. Restrict data to a specific year
4. Resample to uniform 15-minute intervals
5. Preserve missing values (no interpolation)

Why this step is important:
- Ensures consistent temporal resolution
- Removes irregular sampling issues
- Enables reliable rolling statistics
- Missing values are intentionally preserved to avoid introducing artificial bias
"""
def filter_and_resample_data(sensor, year):
    """
    Filters PM2.5 data for a specific sensor and year,
    then resamples it to uniform 15-minute intervals.

    Parameters:
    - sensor_name (str): Target sensor ID
    - data (DataFrame): Raw dataset
    - year (int): Target year

    Returns:
    - DataFrame: Cleaned and resampled dataset
    """

    # --- Convert timestamp ---
    sensor["Timestamp"] = pd.to_datetime(sensor["Timestamp"], errors="coerce")

    # --- Define yearly range ---
    start_date = f"{year}-03-01 00:00:00"
    end_date   = f"{year}-03-08 23:59:59"

    # --- Filter by date ---
    filtered_sensor = sensor.loc[
        (sensor["Timestamp"] >= start_date) &
        (sensor["Timestamp"] <= end_date)
    ]

    # --- Set index for resampling ---
    filtered_sensor = filtered_sensor.set_index("Timestamp")

    # --- Resample to 15-minute intervals (no filling) ---
    # 15-minute resolution balances temporal detail and computational cost
    filtered_sensor = filtered_sensor.resample("15T").asfreq()

    # --- Reset index ---
    filtered_sensor = filtered_sensor.reset_index()

    return filtered_sensor

In [ ]:
"""
Create a complete time index for the entire year
with 15-minute resolution.

Useful for:
- Detecting missing timestamps
- Aligning predictions
"""

start_date = "2025-03-01 00:00:00"
end_date   = "2025-03-08 23:59:00"

time_wholeyear_pm = pd.date_range(
    start=start_date,
    end=end_date,
    freq="15min"
)

MESH_resampled = filter_and_resample_data(MESH, 2025)#
MONITOR_resampled = filter_and_resample_data(MONITOR, 2025)

# Step 2: align to full timeline
MESH_resampled = MESH_resampled.set_index("Timestamp").reindex(time_wholeyear_pm).reset_index()
MONITOR_resampled = MONITOR_resampled.set_index("Timestamp").reindex(time_wholeyear_pm).reset_index()
MESH_resampled = MESH_resampled.rename(columns={"index": "Timestamp"})
MONITOR_resampled = MONITOR_resampled.rename(columns={"index": "Timestamp"})

In [ ]:
"""
Preprocess PM2.5 dataset and generate rolling statistics.

Steps:
1. Ensure datetime format
2. Sort data chronologically
3. Compute short-term smoothing
4. Estimate long-term trend (~1 day)
5. Detrend signal
6. Smooth detrended signal

Why preprocessing is important:
- Reduces high-frequency noise in PM2.5 observations
- Separates long-term trends from short-term fluctuations
- Improves Gaussian Process model stability and predictive performance

Notes:
- 15-minute interval data
- 96 points ≈ 1 day → use 97 (centered window)
"""
def preprocess_sensor(df_sensor):
    df = df_sensor.copy()
    
    df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
    df = df.sort_values("Timestamp").reset_index(drop=True)
    df = df.set_index("Timestamp")

    df["Standardised"] = (df["Value"] - df["Value"].mean())/df["Value"].std()

    return df.reset_index()

# Step 3: preprocess
MESH_processed = preprocess_sensor(MESH_resampled)
MONITOR_processed = preprocess_sensor(MONITOR_resampled)

MESH_processed["Standardised"] = pd.to_numeric(
    MESH_processed["Standardised"],
    errors="coerce"
).fillna(0)

MONITOR_processed["Standardised"] = pd.to_numeric(
    MONITOR_processed["Standardised"],
    errors="coerce"
).fillna(0)

In [ ]:
# TOPLOGICAL DATA ANALYSIS FOR TIME SERIES

# Trying Takens embedding

embedding_dimension = 3 # just testing for visualisation purposes but try 5 next
embedding_time_delay = 6 # 1.5 hours for 15 min interval data
stride = 6

embedder = SingleTakensEmbedding(
    parameters_type="fixed",
    n_jobs=2,
    time_delay=embedding_time_delay,
    dimension=embedding_dimension,
    stride=stride,
)

mesh_embedded = embedder.fit_transform(MESH_processed["Standardised"])
monitor_embedded = embedder.fit_transform(MONITOR_processed["Standardised"])

In [ ]:
plot_point_cloud(mesh_embedded)

In [ ]:
plot_point_cloud(monitor_embedded)

In [ ]:
mesh_embedded = mesh_embedded[None, :, :]
monitor_embedded = monitor_embedded[None, :, :]

# 0 - connected components, 1 - loops, 2 - voids
homology_dimensions = [0, 1, 2]

In [ ]:
mesh_persistence = VietorisRipsPersistence(
    homology_dimensions=homology_dimensions, n_jobs=6
)
print("Persistence diagram for MESH data")
mesh_persistence.fit_transform_plot(mesh_embedded)

In [ ]:
monitor_persistence = VietorisRipsPersistence(
    homology_dimensions=homology_dimensions, n_jobs=6
)
print("Persistence diagram for MONITOR data")
monitor_persistence.fit_transform_plot(monitor_embedded)

In [ ]:
from gtda.diagrams import WassersteinDistance

mesh_diag = mesh_persistence.fit_transform(mesh_embedded)
monitor_diag = monitor_persistence.fit_transform(monitor_embedded)

wasserstein = WassersteinDistance(metric="wasserstein", order=1)

dist = wasserstein.fit_transform([mesh_diag[0], monitor_diag[0]])
print(dist)

# DEFRA sensor comparisons

- Some comparisons between the DEFRA/local automatic sensors and MESH sensors from UO
- Scatter, timeseries, differences, durinal and acf plots to paired sensors close to each other

In [ ]:
DEFRA_2026 = pd.read_csv("2026uptoMay13-PM25-DEFRA.csv")
local_2026 = pd.read_csv("2026uptoMay13-PM25-local.csv")

precision_list = [DEFRA_2026, local_2026]

for df in precision_list:
    date = df['Date'].astype(str)
    time = df['Time'].astype(str)

    mask_24 = time.str.startswith('24:')

    # fix time first
    time_fixed = time.str.replace(r'^24:', '00:', regex=True)

    # combine as strings
    combined = date + ' ' + time_fixed

    # parse AFTER fixing
    ts = pd.to_datetime(combined, dayfirst=True)

    # now shift ONLY those originally with 24:00
    ts = ts + pd.to_timedelta(mask_24.astype(int), unit='D')

    df['Timestamp'] = ts

    df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")

defra_march_2026 = DEFRA_2026[
    (DEFRA_2026["Timestamp"] >= "2026-03-01") &
    (DEFRA_2026["Timestamp"] < "2026-03-31")
]

local_march_2026 = local_2026[
    (local_2026["Timestamp"] >= "2026-03-01") &
    (local_2026["Timestamp"] < "2026-03-31")
]

sensor_locations = (
uo_march_2026[['Sensor_Name', 'Sensor_Centroid_Longitude', 'Sensor_Centroid_Latitude']]
    .dropna()
    .drop_duplicates(subset='Sensor_Name')
    .reset_index(drop=True)
)

mesh_locations = sensor_locations[
    sensor_locations['Sensor_Name'].str.contains('MESH', na=False)
].reset_index(drop=True)

monitor_locations = (
    sensor_locations[sensor_locations['Sensor_Name'].str.contains('MONITOR', na=False)]
    .reset_index(drop=True)
    .rename(columns={'Sensor_Centroid_Longitude': 'Long'})
    .rename(columns={'Sensor_Centroid_Latitude': 'Lat'})
)

defra_locations = (
    defra_march_2026[['Sensor_Name', 'Long', 'Lat']]
    .dropna()
    .drop_duplicates(subset='Sensor_Name')
    .reset_index(drop=True)
)

local_locations = (
    local_march_2026[['Sensor_Name', 'Long', "Lat"]]
    .dropna()
    .drop_duplicates(subset='Sensor_Name')
    .reset_index(drop=True)
)

precision_locations = [defra_locations, local_locations, monitor_locations]


- Some comprisons between the DEFRA/local automatic sensors and MESH sensors from UO
- Scatter, timeseries, differences, durinal and acf plots to paired sensors close to each other

In [ ]:
results = []

for df in precision_locations:
   for _, precision_row in df.iterrows():

        precision_coords = (
            precision_row['Lat'],
            precision_row['Long']
        )

        # compute distance to every sensor
        for _, mesh_row in mesh_locations.iterrows():

            mesh_coords = (
                mesh_row['Sensor_Centroid_Latitude'],
                mesh_row['Sensor_Centroid_Longitude']
            )

            distance_km = geodesic(
                precision_coords,
                mesh_coords
            ).km

            results.append({
                'precision_sensor': precision_row['Sensor_Name'],
                'Mesh_Name': mesh_row['Sensor_Name'],
                'distance_km': distance_km
            })

distance_df = pd.DataFrame(results)

closest_sensors_mesh = (
    distance_df.loc[
        distance_df.groupby('precision_sensor')['distance_km'].idxmin()
    ]
)

print(closest_sensors_mesh.head())

In [ ]:
results = []
defra2_locations = pd.concat(
    [defra_locations, local_locations],
    ignore_index=True
)

for _, defra2_row in defra2_locations.iterrows():

    defra2_coords = (
        defra2_row['Lat'],
        defra2_row['Long']
    )

    for _, monitor_row in monitor_locations.iterrows():

        monitor_coords = (
            monitor_row['Lat'],
            monitor_row['Long']
        )

        distance2_km = geodesic(
            defra2_coords,
            monitor_coords
        ).km

        results.append({
            'defra_sensor': defra2_row['Sensor_Name'],
            'Monitor_Name': monitor_row['Sensor_Name'],
            'distance_km': distance2_km
        })

distance2_df = pd.DataFrame(results)

closest_sensors_monitor = (
    distance2_df.loc[
        distance2_df.groupby('defra_sensor')['distance_km'].idxmin()
    ]
)

print(closest_sensors_monitor.head())

In [ ]:
defra_gdf = gpd.GeoDataFrame(
    defra_locations,
    geometry=gpd.points_from_xy(defra_locations['Long'], defra_locations['Lat']),
    crs="EPSG:4326"
)

monitor_gdf = gpd.GeoDataFrame(
    monitor_locations,
    geometry=gpd.points_from_xy(monitor_locations['Long'], monitor_locations['Lat']),
    crs="EPSG:4326"
)

defra_gdf = defra_gdf.to_crs(epsg=3857)
monitor_gdf = monitor_gdf.to_crs(epsg=3857)

fig, ax = plt.subplots(figsize=(10, 10))

for _, row in closest_sensors_monitor.iterrows():

    d = defra_gdf.(row['defra_sensor'])
    m = monitor_lookup.get(row['Monitor_Name'])

    ax.plot([d.x, m.x], [d.y, m.y],
        color='black',
        linewidth=1,
        alpha=0.6,
        zorder=3)

ax.set_xlim(defra_gdf.total_bounds[0], defra_gdf.total_bounds[2])
ax.set_ylim(defra_gdf.total_bounds[1], defra_gdf.total_bounds[3])

ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)

ax.set_axis_off()
ax.set_title("Defra → Nearest Monitor Sensor Network")

plt.show()

In [ ]:
results = []

for _, mesh_row in mesh_locations.iterrows():

    mesh_coords = (
        mesh_row['Sensor_Centroid_Latitude'],
        mesh_row['Sensor_Centroid_Longitude']
    )

    for _, monitor_row in monitor_locations.iterrows():

        monitor_coords = (
            monitor_row['Lat'],
            monitor_row['Long']
        )

        distance3_km = math.dist(mesh_coords, monitor_coords)

        results.append({
            'Mesh_sensor': mesh_row['Sensor_Name'],
            'Monitor_Name': monitor_row['Sensor_Name'],
            'distance_km': distance3_km
        })

distance3_df = pd.DataFrame(results)

closest_sensors_mesh_monitor = (
    distance3_df.loc[
        distance3_df.groupby('Mesh_sensor')['distance_km'].idxmin()
    ]
)

print(closest_sensors_mesh_monitor.head())

In [ ]:
print(defra_march_2026.columns.tolist())
print(uo_march_2026.columns.tolist())

defra_names = set(defra_locations['Sensor_Name'].unique())
local_names = set(local_locations['Sensor_Name'].unique())
monitor_names = set(monitor_locations['Sensor_Name'].unique()) 

# ONLY RERUN FOR WHOLE MONTH

In [ ]:
for _, row in closest_sensors_mesh.iterrows():

    precision_name = row['precision_sensor']
    mesh_name = row['Mesh_Name']

    print(precision_name, 'and', mesh_name)
    print("Distance:", row['distance_km'], "km")

    if precision_name in defra_names:
        precision_df = defra_march_2026
    elif precision_name in local_names:
        precision_df = local_march_2026
    elif precision_name in monitor_names:
        precision_df = (uo_march_2026[uo_march_2026['Sensor_Name'].str.contains('MONITOR', na=False)]
            .reset_index(drop=True)
            .rename(columns={'Value': 'PM2.5'
                             })
                             ) 

    # subset
    precision = (
        precision_df[
            precision_df['Sensor_Name'] == precision_name
        ][['Timestamp', 'PM2.5']]
        .sort_values('Timestamp')
        .rename(columns={'PM2.5': 'precision_pm25'})
    )

    precision['precision_pm25'] = pd.to_numeric(
        precision['precision_pm25'],
        errors='coerce'
    )

    # matching mesh subset
    mesh = (
        uo_march_2026[
            uo_march_2026['Sensor_Name'] == mesh_name
        ][['Timestamp', 'Value']]
        .sort_values('Timestamp')
        .rename(columns={'Value': 'mesh_pm25'})
    )

    mesh['mesh_pm25'] = pd.to_numeric(
        mesh['mesh_pm25'], 
        errors='coerce')

    # nearest timestamp match
    merged = pd.merge_asof(
        precision,
        mesh,
        on='Timestamp',
        direction='nearest',
        tolerance=pd.Timedelta('1min')
    )

    merged = merged.dropna(subset=['precision_pm25', 'mesh_pm25'])

    corr = merged[['precision_pm25', 'mesh_pm25']].corr().iloc[0, 1]
    print(f"Correlation ({precision_name} vs {mesh_name}): {corr:.3f}")

    print(f'Matched rows: {len(merged)}')

    if merged.empty:
        print("No matched timestamps")
        continue

    # force NEW figure
    fig, ax = plt.subplots(figsize=(6, 6))

    time_num = (
    merged['Timestamp'].dt.hour
    + merged['Timestamp'].dt.minute / 60
    + merged['Timestamp'].dt.second / 3600)

    pm25_plt = ax.scatter(
        merged['precision_pm25'],
        merged['mesh_pm25'],
        alpha=0.5,
        c=time_num,
        s = 20
    )

    cb = plt.colorbar(pm25_plt, format=lambda t, pos: f'{int(t):02d}:{int((t % 1) * 60):02d}')

    cb.set_label('Time of day') 

    # 1:1 line
    lims = [
    merged[['precision_pm25', 'mesh_pm25']].min().min(),
    merged[['precision_pm25', 'mesh_pm25']].max().max()
]

    ax.plot(lims, lims)

    ax.set_xlabel('Precision PM2.5')
    ax.set_ylabel('Mesh PM2.5')
    ax.set_title(f'{precision_name}\nvs\n{mesh_name}')
    ax.grid(True)

    plt.savefig(f'March_2026_Defra_vs_MonUO\precision_to_mesh\{precision_name}_{mesh_name}.pdf')
    plt.show()
    plt.close(fig)

In [ ]:
summary_results = []

for _, row in closest_sensors_monitor.iterrows():

    precision_name = row['defra_sensor']
    monitor_name = row['Monitor_Name']

    print(precision_name, 'and', monitor_name)
    print("Distance:", row['distance_km'], "km")

    if precision_name in defra_names:
        precision_df = defra_march_2026
    elif precision_name in local_names:
        precision_df = local_march_2026

    # subset
    precision = (
        precision_df[
            precision_df['Sensor_Name'] == precision_name
        ][['Timestamp', 'PM2.5']]
        .sort_values('Timestamp')
        .rename(columns={'PM2.5': 'precision_pm25'})
    )

    precision['precision_pm25'] = pd.to_numeric(
        precision['precision_pm25'],
        errors='coerce'
    )

    monitor = (
        uo_march_2026[
            uo_march_2026['Sensor_Name'] == monitor_name
        ][['Timestamp', 'Value']]
        .sort_values('Timestamp')
        .rename(columns={'Value': 'monitor_pm25'})
    )

    monitor['monitor_pm25'] = pd.to_numeric(
        monitor['monitor_pm25'], 
        errors='coerce')

    # nearest timestamp match
    merged = pd.merge_asof(
        precision,
        monitor,
        on='Timestamp',
        direction='nearest',
        tolerance=pd.Timedelta('1min')
    )

    merged = merged.dropna(subset=['precision_pm25', 'monitor_pm25'])

    # Pearson correlation
    pearson_corr = merged[['precision_pm25', 'monitor_pm25']].corr(method='pearson').iloc[0, 1]

    # Spearman correlation
    spearman_corr = merged[['precision_pm25', 'monitor_pm25']].corr(method='spearman').iloc[0, 1]

    summary_results.append({
    'precision_sensor': precision_name,
    'monitor_name': monitor_name,
    'distance_km': row['distance_km'],
    'pearson_corr': pearson_corr,
    'spearman_corr': spearman_corr,
    })  


    if merged.empty:
        print("No matched timestamps")
        continue

    # force NEW figure
    fig, ax = plt.subplots(figsize=(6, 6))

    time_num = (
    merged['Timestamp'].dt.hour
    + merged['Timestamp'].dt.minute / 60
    + merged['Timestamp'].dt.second / 3600)

    pm25_plt = ax.scatter(
        merged['precision_pm25'],
        merged['monitor_pm25'],
        alpha=0.5,
        c=time_num,
        s = 20
    )

    cb = plt.colorbar(pm25_plt, format=lambda t, pos: f'{int(t):02d}:{int((t % 1) * 60):02d}')

    cb.set_label('Time of day')

    # 1:1 line
    lims = [
    merged[['precision_pm25', 'monitor_pm25']].min().min(),
    merged[['precision_pm25', 'monitor_pm25']].max().max()
]

    ax.plot(lims, lims)

    ax.set_xlabel('Defra PM2.5')
    ax.set_ylabel('Monitor (UO) PM2.5')
    ax.set_title(f'{precision_name}\nvs\n{monitor_name}')
    ax.grid(True)

    save_dir = Path(f"March_2026_Defra_vs_MonUO\{precision_name}_vs_{monitor_name}")
    save_dir.mkdir(parents=True, exist_ok=True)

    plt.savefig(save_dir/"scatter_hourly.pdf")
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(12, 4))

    ax.plot(
        merged['Timestamp'],
        merged['precision_pm25'],
        label='Defra'
    )

    ax.plot(
        merged['Timestamp'],
        merged['monitor_pm25'],
        label='Monitor (UO)'
    )

    ax.set_xlabel('Time')
    ax.set_ylabel('PM2.5')
    ax.set_title(f'{precision_name} vs {monitor_name}')
    ax.legend()
    ax.grid(True)

    plt.savefig(save_dir/"timeseries.pdf")
    plt.show()

    window = 24  # hourly points

    rolling_corr = (
        merged['precision_pm25']
        .rolling(window)
        .corr(merged['monitor_pm25'])
    )

    plt.figure(figsize=(10,4))

    plt.plot(
        merged['Timestamp'],
        rolling_corr
    )

    plt.axhline(0, linestyle='--')

    plt.ylabel('Rolling correlation (24h)')
    plt.xlabel('Time')
    plt.title(
        f'Rolling correlation (24h)\n'
        f'{precision_name} vs {monitor_name}'
    )

    plt.grid(True)
    plt.savefig(save_dir/"rolling_corr_24h.pdf")
    plt.show()

    merged['difference'] = (
    merged['monitor_pm25']
    - merged['precision_pm25']
)

    plt.figure(figsize=(12,4))

    plt.plot(
        merged['Timestamp'],
        merged['difference']
    )

    plt.axhline(0)

    plt.ylabel('Monitor (UO) - Defra')
    plt.xlabel('Time')
    plt.title(
        f'Difference through time\n'
        f'{precision_name} vs {monitor_name}'
    )

    plt.grid(True)
    plt.savefig(save_dir/"time_diff.pdf")
    plt.show()

    merged['hour'] = merged['Timestamp'].dt.hour

    hourly = (
        merged
        .groupby('hour')[
            ['precision_pm25', 'monitor_pm25']
        ]
        .mean()
    )

    plt.figure(figsize=(8,4))

    plt.plot(hourly['precision_pm25'], label='Defra')
    plt.plot(hourly['monitor_pm25'], label='Monitor (UO)')
    plt.ylabel('Mean PM2.5')
    plt.title(
        f'Diurnal cycle\n'
        f'{precision_name} vs {monitor_name}'
    )
    plt.grid(True)
    plt.legend()

    plt.savefig(save_dir/"diurnal.pdf")
    plt.show()

    precision_acf = acf(
        merged['precision_pm25'].dropna(),
        nlags=48
        )

    monitor_acf = acf(
        merged['monitor_pm25'].dropna(),
        nlags=48
    )

    plt.figure(figsize=(8,5))

    plt.plot(precision_acf, label='Defra')
    plt.plot(monitor_acf, label='Monitor (UO)')

    plt.xlabel('Lag')
    plt.ylabel('Autocorrelation')
    plt.title(
        f'ACF comparison\n'
        f'{precision_name} vs {monitor_name}'
    )

    plt.legend()
    plt.grid(True)

    plt.savefig(save_dir/"acf.pdf")
    plt.show()

In [ ]:
summary_df = pd.DataFrame(summary_results)

summary_df = summary_df[[
    'precision_sensor',
    'monitor_name',
    'distance_km',
    'pearson_corr',
    'spearman_corr'
]]

summary_df = summary_df.sort_values(
    by='pearson_corr',
    ascending=False
)

summary_df[['distance_km', 'pearson_corr', 'spearman_corr']] = (
    summary_df[['distance_km', 'pearson_corr', 'spearman_corr']].round(3)
)

summary_df

summary_df.to_csv("March_2026_Defra_vs_MonUO/defra_monitor_distances_corr.csv", index=False)

# ONLY RERUN FOR WEEK

In [ ]:
summary_results = []

for _, row in closest_sensors_mesh_monitor.iterrows():

    mesh_name = row['Mesh_sensor']
    monitor_name = row['Monitor_Name']

    print(mesh_name, 'and', monitor_name)
    print("Distance:", row['distance_km'], "km")

    mesh = (
        uo_march_2026[
            uo_march_2026['Sensor_Name'] == mesh_name
        ][['Timestamp', 'Value']]
        .sort_values('Timestamp')
        .rename(columns={'Value': 'mesh_pm25'})
    )

    mesh['mesh_pm25'] = pd.to_numeric(
        mesh['mesh_pm25'],
        errors='coerce'
    )

    monitor = (
        uo_march_2026[
            uo_march_2026['Sensor_Name'] == monitor_name
        ][['Timestamp', 'Value']]
        .sort_values('Timestamp')
        .rename(columns={'Value': 'monitor_pm25'})
    )

    monitor['monitor_pm25'] = pd.to_numeric(
        monitor['monitor_pm25'], 
        errors='coerce')

    # nearest timestamp match
    merged = pd.merge_asof(
        mesh,
        monitor,
        on='Timestamp',
        direction='nearest',
        tolerance=pd.Timedelta('1min')
    )

    merged = merged.dropna(subset=['mesh_pm25', 'monitor_pm25'])

    # Pearson correlation
    pearson_corr = merged[['mesh_pm25', 'monitor_pm25']].corr(method='pearson').iloc[0, 1]

    # Spearman correlation
    spearman_corr = merged[['mesh_pm25', 'monitor_pm25']].corr(method='spearman').iloc[0, 1]

    summary_results.append({
    'mesh_sensor': mesh_name,
    'monitor_name': monitor_name,
    'distance_km': row['distance_km'],
    'pearson_corr': pearson_corr,
    'spearman_corr': spearman_corr,
    })
    
    fig, ax = plt.subplots(figsize=(12, 4))

    ax.plot(
        merged['Timestamp'],
        merged['mesh_pm25'],
        label='Mesh (UO)'
    )

    ax.plot(
        merged['Timestamp'],
        merged['monitor_pm25'],
        label='Monitor (UO)'
    )

    ax.set_xlabel('Time')
    ax.set_ylabel('PM2.5')
    ax.set_title(f'{mesh_name} vs {monitor_name}')
    ax.legend()
    ax.grid(True)

    save_dir = Path(f"Week_March_2026_Monitor_Mesh\{mesh_name}_vs_{monitor_name}")
    save_dir.mkdir(parents=True, exist_ok=True)

    plt.savefig(save_dir/"timeseries.pdf")
    plt.show()

    window = 24  # hourly points

    rolling_corr = (
        merged['mesh_pm25']
        .rolling(window)
        .corr(merged['monitor_pm25'])
    )

    plt.figure(figsize=(10,4))

    plt.plot(
        merged['Timestamp'],
        rolling_corr
    )

    plt.axhline(0, linestyle='--')

    plt.ylabel('Rolling correlation (24h)')
    plt.xlabel('Time')
    plt.title(
        f'Rolling correlation (24h)\n'
        f'{mesh_name} vs {monitor_name}'
    )

    plt.savefig(save_dir/"rolling_corr_24h.pdf")
    plt.grid(True)
    plt.show()

    merged['difference'] = (
    merged['monitor_pm25']
    - merged['mesh_pm25']
)

    plt.figure(figsize=(12,4))

    plt.plot(
        merged['Timestamp'],
        merged['difference']
    )

    plt.axhline(0)

    plt.ylabel('Monitor (UO) - Mesh (UO)')
    plt.xlabel('Time')
    plt.title(
        f'Difference through time\n'
        f'{mesh_name} vs {monitor_name}'
    )

    plt.grid(True)

    plt.savefig(save_dir/"time_diff.pdf")
    plt.show()

    merged['hour'] = merged['Timestamp'].dt.hour

    hourly = (
        merged
        .groupby('hour')[
            ['mesh_pm25', 'monitor_pm25']
        ]
        .mean()
    )

    plt.figure(figsize=(8,4))

    plt.plot(hourly['mesh_pm25'], label='Mesh (UO)')
    plt.plot(hourly['monitor_pm25'], label='Monitor (UO)')
    plt.ylabel('Mean PM2.5')
    plt.title(
        f'Diurnal cycle\n'
        f'{mesh_name} vs {monitor_name}'
    )
    plt.grid(True)
    plt.legend()

    plt.savefig(save_dir/"diurnal.pdf")
    plt.show()

    mesh_acf = acf(
        merged['mesh_pm25'].dropna(),
        nlags=48
        )

    monitor_acf = acf(
        merged['monitor_pm25'].dropna(),
        nlags=48
    )

    plt.figure(figsize=(8,5))

    plt.plot(mesh_acf, label='Mesh (UO)')
    plt.plot(monitor_acf, label='Monitor (UO)')

    plt.xlabel('Lag')
    plt.ylabel('Autocorrelation')
    plt.title(
        f'ACF comparison\n'
        f'{mesh_name} vs {monitor_name}'
    )

    plt.legend()
    plt.grid(True)

    plt.savefig(save_dir/"acf.pdf")
    plt.show()

summary_df = pd.DataFrame(summary_results)

summary_df = summary_df[[
    'mesh_sensor',
    'monitor_name',
    'distance_km',
    'pearson_corr',
    'spearman_corr'
]]

summary_df = summary_df.sort_values(
    by='pearson_corr',
    ascending=False
)

summary_df[['distance_km', 'pearson_corr', 'spearman_corr']] = (
    summary_df[['distance_km', 'pearson_corr', 'spearman_corr']].round(3)
)

summary_df

summary_df.to_csv("Week_March_2026_Monitor_Mesh/mesh_monitor_distances_corr.csv", index=False)


# NO comparisons with PM2.5

Come back to this later, data for UO not wanting to download for NOx or NO2, just compare defra and local for now

In [ ]:
df = uo_pyfetch.get_sensor_data(
            start=datetime.datetime(2026, 1, 1),
            end=datetime.datetime(2026, 1, 31, 23, 59, 59),
            variables=["NOx"],
            limit=-1
        )

In [ ]:
dfs = []

for month in range(1, 6):
    start = datetime.datetime(2026, month, 1)

    if month == 5:
        final = datetime.datetime(2026, month, 27, 23, 59, 59)
    else:
        final = datetime.datetime(2026, month + 1, 1) - datetime.timedelta(seconds=1)

    print(f"Fetching {start:%Y-%m}")

    while start < final:
        end = min(start + datetime.timedelta(days=7), final)

        try:
            df = uo_pyfetch.get_sensor_data(
                start=start,
                end=end,
                variables=["NO2"],
                limit=-1
            )
            dfs.append(df)

        except Exception as e:
            print(f"Failed for {month}: {e}")
        start=end

NOx_2026 = pd.concat(dfs, ignore_index=True)

NOx_2026.to_csv('2026uptoMay27-NO2-UO.csv')

In [ ]:
NOx_2026 = pd.read_csv("2026uptoMay27-NOx-UO.csv")

NOx_2026["Timestamp"] = pd.to_datetime(NOx_2026["Timestamp"], errors="coerce")

uo_march_2026_NOx = NOx_2026[
    (NOx_2026["Timestamp"] >= "2026-03-23") &
    (NOx_2026["Timestamp"] < "2026-03-29")
]


In [ ]:
DEFRA_2026_NO = pd.read_csv("2026uptoMay13-NO-DEFRA.csv")
local_2026_NO = pd.read_csv("2026uptoMay13-NO-local.csv")

precision_list = [DEFRA_2026_NO, local_2026_NO]

for df in precision_list:
    date = df['Date'].astype(str)
    time = df['Time'].astype(str)

    mask_24 = time.str.startswith('24:')

    # fix time first
    time_fixed = time.str.replace(r'^24:', '00:', regex=True)

    # combine as strings
    combined = date + ' ' + time_fixed

    # parse AFTER fixing
    ts = pd.to_datetime(combined, dayfirst=True)

    # now shift ONLY those originally with 24:00
    ts = ts + pd.to_timedelta(mask_24.astype(int), unit='D')

    df['Timestamp'] = ts

    df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")

defra_march_2026_NO = DEFRA_2026_NO[
    (DEFRA_2026_NO["Timestamp"] >= "2026-03-23") &
    (DEFRA_2026_NO["Timestamp"] < "2026-03-29")
]

local_march_2026_NO = local_2026_NO[
    (local_2026_NO["Timestamp"] >= "2026-03-23") &
    (local_2026_NO["Timestamp"] < "2026-03-29")
]


In [ ]:
summary_results = []

for name in defra_names:

    print(name)

    pm25 = (
        defra_march_2026[
            defra_march_2026['Sensor_Name'] == name
        ][['Timestamp', 'PM2.5']]
        .sort_values('Timestamp')
    )

    pm25['PM2.5'] = pd.to_numeric(
        pm25['PM2.5'],
        errors='coerce'
    )

    nox = (
        defra_march_2026_NO[
            defra_march_2026_NO['Sensor_Name'] == name
        ][['Timestamp', 'NOx', 'Time', 'Date']]
        .sort_values('Timestamp')
    )

    nox['NOx'] = pd.to_numeric(
        nox['NOx'],
        errors='coerce'
    )

    no2 = (
        defra_march_2026_NO[
            defra_march_2026_NO['Sensor_Name'] == name
        ][['Timestamp', 'NO2', 'Time', 'Date']]
        .sort_values('Timestamp')
    )

    no2['NO2'] = pd.to_numeric(
        no2['NO2'],
        errors='coerce'
    )

    # nearest timestamp match
    merged_nox = pd.merge_asof(
        pm25,
        nox,
        on='Timestamp',
        direction='nearest',
        tolerance=pd.Timedelta('1min')
    )

    merged_nox = merged_nox.dropna(subset=['PM2.5', 'NOx'])

    merged_no2 = pd.merge_asof(
        pm25,
        no2,
        on='Timestamp',
        direction='nearest',
        tolerance=pd.Timedelta('1min')
    )

    merged_no2 = merged_no2.dropna(subset=['PM2.5', 'NO2'])

    # Pearson correlation
    pearson_corr_nox = merged_nox[['PM2.5', 'NOx']].corr(method='pearson').iloc[0, 1]
    pearson_corr_no2 = merged_no2[['PM2.5', 'NO2']].corr(method='pearson').iloc[0, 1]

    # Spearman correlation
    spearman_corr_nox = merged_nox[['PM2.5', 'NOx']].corr(method='spearman').iloc[0, 1]
    spearman_corr_no2 = merged_no2[['PM2.5', 'NO2']].corr(method='spearman').iloc[0, 1]

    summary_results.append({
    'defra_sensor': name,
    'pearson_corr_nox': pearson_corr_nox,
    'pearson_corr_no2': pearson_corr_no2,
    'spearman_corr_nox': spearman_corr_nox,
    'spearman_corr_no2': spearman_corr_no2,
    })

    fig, ax = plt.subplots(figsize=(6, 6))

    time_num = (
    merged_nox['Timestamp'].dt.hour
    + merged_nox['Timestamp'].dt.minute / 60
    + merged_nox['Timestamp'].dt.second / 3600)

    nox_plt = ax.scatter(
        merged_nox['PM2.5'],
        merged_nox['NOx'],
        alpha=0.5,
        c=time_num,
        s = 20
    )

    cb = plt.colorbar(nox_plt, format=lambda t, pos: f'{int(t):02d}:{int((t % 1) * 60):02d}')

    cb.set_label('Time of day') 

    # 1:1 line
    lims = [
    merged_nox[['PM2.5', 'NOx']].min().min(),
    merged_nox[['PM2.5', 'NOx']].max().max()
]

    ax.plot(lims, lims)

    ax.set_xlabel('PM2.5')
    ax.set_ylabel('NOx')
    ax.set_title(f'{name}')
    ax.grid(True)

    plt.savefig(f'Week_March_2026_NO/{name}_PM25_NOx.pdf')
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(6, 6))

    time_num = (
    merged_no2['Timestamp'].dt.hour
    + merged_no2['Timestamp'].dt.minute / 60
    + merged_no2['Timestamp'].dt.second / 3600)

    no2_plt = ax.scatter(
        merged_no2['PM2.5'],
        merged_no2['NO2'],
        alpha=0.5,
        c=time_num,
        s = 20
    )

    cb = plt.colorbar(no2_plt, format=lambda t, pos: f'{int(t):02d}:{int((t % 1) * 60):02d}')

    cb.set_label('Time of day') 

    # 1:1 line
    lims = [
    merged_no2[['PM2.5', 'NO2']].min().min(),
    merged_no2[['PM2.5', 'NO2']].max().max()
]

    ax.plot(lims, lims)

    ax.set_xlabel('PM2.5')
    ax.set_ylabel('NO2')
    ax.set_title(f'{name}')
    ax.grid(True)

    plt.savefig(f'Week_March_2026_NO/{name}_PM25_NO2.pdf')
    plt.show()
    plt.close(fig)
    





In [ ]:
summary_results = []

for name in local_names:

    print(name)

    pm25 = (
        local_march_2026[
            local_march_2026['Sensor_Name'] == name
        ][['Timestamp', 'PM2.5']]
        .sort_values('Timestamp')
    )

    pm25['PM2.5'] = pd.to_numeric(
        pm25['PM2.5'],
        errors='coerce'
    )

    no2 = (
        local_march_2026_NO[
            local_march_2026_NO['Sensor_Name'] == name
        ][['Timestamp', 'NO2', 'Time', 'Date']]
        .sort_values('Timestamp')
    )

    no2['NO2'] = pd.to_numeric(
        no2['NO2'],
        errors='coerce'
    )

    merged_no2 = pd.merge_asof(
        pm25,
        no2,
        on='Timestamp',
        direction='nearest',
        tolerance=pd.Timedelta('1min')
    )

    merged_no2 = merged_no2.dropna(subset=['PM2.5', 'NO2'])

    # Pearson correlation
    pearson_corr_no2 = merged_no2[['PM2.5', 'NO2']].corr(method='pearson').iloc[0, 1]

    # Spearman correlation
    spearman_corr_no2 = merged_no2[['PM2.5', 'NO2']].corr(method='spearman').iloc[0, 1]

    summary_results.append({
    'defra_sensor': name,
    'pearson_corr_nox': pearson_corr_nox,
    'pearson_corr_no2': pearson_corr_no2,
    'spearman_corr_nox': spearman_corr_nox,
    'spearman_corr_no2': spearman_corr_no2,
    })

    fig, ax = plt.subplots(figsize=(6, 6))

    time_num = (
    merged_no2['Timestamp'].dt.hour
    + merged_no2['Timestamp'].dt.minute / 60
    + merged_no2['Timestamp'].dt.second / 3600)

    no2_plt = ax.scatter(
        merged_no2['PM2.5'],
        merged_no2['NO2'],
        alpha=0.5,
        c=time_num,
        s = 20
    )

    cb = plt.colorbar(no2_plt, format=lambda t, pos: f'{int(t):02d}:{int((t % 1) * 60):02d}')

    cb.set_label('Time of day') 

    # 1:1 line
    lims = [
    merged_no2[['PM2.5', 'NO2']].min().min(),
    merged_no2[['PM2.5', 'NO2']].max().max()
]

    ax.plot(lims, lims)

    ax.set_xlabel('PM2.5')
    ax.set_ylabel('NO2')
    ax.set_title(f'{name}')
    ax.grid(True)

    plt.savefig(f'Week_March_2026_NO/{name}_PM25_NO2.pdf')
    plt.show()
    plt.close(fig)

In [ ]:
summary_results = []

for name in monitor_names:

    print(name)

    pm25 = (
        uo_march_2026[
            uo_march_2026['Sensor_Name'] == name
        ][['Timestamp', 'Value']]
        .sort_values('Timestamp')
        .rename(columns={'Value': 'PM2.5'})
    )

    pm25['PM2.5'] = pd.to_numeric(
        pm25['PM2.5'],
        errors='coerce'
    )

    nox = (
        uo_march_2026_NOx[
            uo_march_2026_NOx['Sensor_Name'] == name
        ][['Timestamp', 'Value']]
        .sort_values('Timestamp')
        .rename(columns={'Value': 'NOx'})
    )

    nox['NOx'] = pd.to_numeric(
        nox['NOx'],
        errors='coerce'
    )

    merged_nox = pd.merge_asof(
        pm25,
        nox,
        on='Timestamp',
        direction='nearest',
        tolerance=pd.Timedelta('1min')
    )

    merged_nox = merged_nox.dropna(subset=['PM2.5', 'NOx'])

    # Pearson correlation
    pearson_corr_nox = merged_nox[['PM2.5', 'NOx']].corr(method='pearson').iloc[0, 1]

    # Spearman correlation
    spearman_corr_nox = merged_nox[['PM2.5', 'NOx']].corr(method='spearman').iloc[0, 1]

    summary_results.append({
    'defra_sensor': name,
    'pearson_corr_nox': pearson_corr_nox,
    'spearman_corr_nox': spearman_corr_nox
    })

    fig, ax = plt.subplots(figsize=(6, 6))

    time_num = (
    merged_nox['Timestamp'].dt.hour
    + merged_nox['Timestamp'].dt.minute / 60
    + merged_nox['Timestamp'].dt.second / 3600)

    nox_plt = ax.scatter(
        merged_nox['PM2.5'],
        merged_nox['NOx'],
        alpha=0.5,
        c=time_num,
        s = 20
    )

    cb = plt.colorbar(nox_plt, format=lambda t, pos: f'{int(t):02d}:{int((t % 1) * 60):02d}')

    cb.set_label('Time of day') 

    # 1:1 line
    lims = [
    merged_nox[['PM2.5', 'NOx']].min().min(),
    merged_nox[['PM2.5', 'NOx']].max().max()
]

    ax.plot(lims, lims)

    ax.set_xlabel('PM2.5')
    ax.set_ylabel('NOx')
    ax.set_title(f'{name}')
    ax.grid(True)

    plt.savefig(f'Week_March_2026_NO/{name}_PM25_NOx.pdf')
    plt.show()
    plt.close(fig)

# Wind direction and speed comparisons with PM2.5

In [10]:
west_denton_jan_2025 = uo_pyfetch.get_sensor_data_by_name(
        'PER_EMLFLOOD_UO-WDENTONFS',
        start=datetime.datetime(2025, 1, 1),
        end=datetime.datetime(2025, 1, 7),
        variables=["Wind Direction", "Wind Speed"],
        limit=-1
    )

birtley_jan_2025 = uo_pyfetch.get_sensor_data_by_name(
        'PER_EMLFLOOD_UO-BIRTLEYFS',
        start=datetime.datetime(2025, 1, 1),
        end=datetime.datetime(2025, 1, 7),
        variables=["Wind Direction", "Wind Speed"],
        limit=-1
    )

In [2]:
west_denton = []
birtley = []

for month in range(1, 13):
    start = datetime.datetime(2025, month, 1)

    if month == 12:
        final = datetime.datetime(2025, month, 31, 23, 59, 59)
    else:
        final = datetime.datetime(2025, month + 1, 1) - datetime.timedelta(seconds=1)

    print(f"Fetching {start:%Y-%m}")

    while start < final:
        end = min(start + datetime.timedelta(days=7), final)

        try:
            wd = uo_pyfetch.get_sensor_data_by_name(
            'PER_EMLFLOOD_UO-WDENTONFS',
            start=start,
            end=end,
            variables=["Wind Direction", "Wind Speed"],
            limit=-1
            )
            west_denton.append(wd)

            birt = uo_pyfetch.get_sensor_data_by_name(
            'PER_EMLFLOOD_UO-BIRTLEYFS',
            start=start,
            end=end,
            variables=["Wind Direction", "Wind Speed"],
            limit=-1
            )
            birtley.append(birt)

        except Exception as e:
            print(f"Failed for {month}: {e}")
        start=end

westdenton_wind_2025 = pd.concat(west_denton, ignore_index=True)
birtley_wind_2025 = pd.concat(birtley, ignore_index=True)

westdenton_wind_2025.to_csv('2025-wind-UO_westdenton.csv')
birtley_wind_2025.to_csv('2025-wind-UO_birtley.csv')

Fetching 2025-01
Fetching 2025-02
Fetching 2025-03
Fetching 2025-04
Fetching 2025-05
Fetching 2025-06
Fetching 2025-07
Fetching 2025-08
Fetching 2025-09
Fetching 2025-10
Fetching 2025-11
Fetching 2025-12


In [ ]:
uo_2026_wind = pd.read_csv("2026uptoMay27-wind-UO.csv")

uo_2026_wind["Timestamp"] = pd.to_datetime(uo_2026_wind["Timestamp"], errors="coerce")

#uo_march_2026_wind = uo_2026_wind[
#    (uo_2026_wind["Timestamp"] >= "2026-03-01") &
#    (uo_2026_wind["Timestamp"] < "2026-03-31")
#]

In [ ]:
summary_results = []

for name in monitor_names:

    print(name)

    pm25 = (
        PM25_2026[
            PM25_2026['Sensor_Name'] == name
        ][['Timestamp', 'Value']]
        .sort_values('Timestamp')
        .rename(columns={'Value': 'PM2.5'})
    )

    pm25['PM2.5'] = pd.to_numeric(
        pm25['PM2.5'],
        errors='coerce'
    )

    # Wind speed
    WS = (
    uo_2026_wind.loc[
        (uo_2026_wind['Sensor_Name'] == name) &
        (uo_2026_wind['Variable'] == 'Wind Speed'),
        ['Timestamp', 'Value']
    ]
    .sort_values('Timestamp')
    .rename(columns={'Value': 'WS'})
    )

    WS['WS'] = pd.to_numeric(
        WS['WS'],
        errors='coerce'
    )
    
    WD = (
    uo_2026_wind.loc[
        (uo_2026_wind['Sensor_Name'] == name) &
        (uo_2026_wind['Variable'] == 'Wind Direction'),
        ['Timestamp', 'Value']
    ]
    .sort_values('Timestamp')
    .rename(columns={'Value': 'WD'})
)

    WD['WD'] = pd.to_numeric(
        WD['WD'],
        errors='coerce'
    )

    merged_WS = pd.merge_asof(
        pm25,
        WS,
        on='Timestamp',
        direction='nearest',
        tolerance=pd.Timedelta('1min')
    )

    merged_WS = merged_WS.dropna(subset=['PM2.5', 'WS'])

    merged_WD = pd.merge_asof(
        pm25,
        WD,
        on='Timestamp',
        direction='nearest',
        tolerance=pd.Timedelta('1min')
    )

    merged_WD = merged_WD.dropna(subset=['PM2.5', 'WD'])

    # Pearson correlation
    pearson_corr_WS = merged_WS[['PM2.5', 'WS']].corr(method='pearson').iloc[0, 1]
    pearson_corr_WD = merged_WD[['PM2.5', 'WD']].corr(method='pearson').iloc[0, 1]

    # Spearman correlation
    spearman_corr_WS = merged_WS[['PM2.5', 'WS']].corr(method='spearman').iloc[0, 1]
    spearman_corr_WD = merged_WD[['PM2.5', 'WD']].corr(method='spearman').iloc[0, 1]

    summary_results.append({
    'defra_sensor': name,
    'pearson_corr_WS': pearson_corr_WS,
    'pearson_corr_WD': pearson_corr_WD,
    'spearman_corr_WS': spearman_corr_WS,
    'spearman_corr_WD': spearman_corr_WD,
    })

    fig, ax = plt.subplots(figsize=(6, 6))

    time_num_WS = (
    merged_WS['Timestamp'].dt.hour
    + merged_WS['Timestamp'].dt.minute / 60
    + merged_WS['Timestamp'].dt.second / 3600)

    WS_plt = ax.scatter(
        merged_WS['PM2.5'],
        merged_WS['WS'],
        alpha=0.5,
        c=time_num_WS,
        s = 20
    )

    cb = plt.colorbar(WS_plt, format=lambda t, pos: f'{int(t):02d}:{int((t % 1) * 60):02d}')

    cb.set_label('Time of day') 

    ax.set_xlabel('PM2.5')
    ax.set_ylabel('Wind Speed')
    ax.set_title(f'{name}')
    ax.grid(True)

    plt.savefig(f'March_2026_wind/{name}_PM25_WS.pdf')
    plt.show()
    plt.close(fig)


    fig, ax = plt.subplots(figsize=(6, 6))

    time_num_WD = (
    merged_WD['Timestamp'].dt.hour
    + merged_WD['Timestamp'].dt.minute / 60
    + merged_WD['Timestamp'].dt.second / 3600)

    WD_plt = ax.scatter(
        merged_WD['PM2.5'],
        merged_WD['WD'],
        alpha=0.5,
        c=time_num_WD,
        s = 20
    )

    cb = plt.colorbar(WD_plt, format=lambda t, pos: f'{int(t):02d}:{int((t % 1) * 60):02d}')

    cb.set_label('Time of day') 

    ax.set_xlabel('PM2.5')
    ax.set_ylabel('Wind Direction')
    ax.set_title(f'{name}')
    ax.set_ylim(0,360)
    ax.grid(True)

    plt.savefig(f'March_2026_wind/{name}_PM25_WD.pdf')
    plt.show()
    plt.close(fig)

# Altitude

In [ ]:
dem = rasterio.open("Newcastle_altitude/combined_altitude_raster_wgs84.tif")

# Temperature

In [ ]:
west_denton = []
birtley = []

for month in range(1, 13):
    start = datetime.datetime(2025, month, 1)

    if month == 12:
        final = datetime.datetime(2025, month, 31, 23, 59, 59)
    else:
        final = datetime.datetime(2025, month + 1, 1) - datetime.timedelta(seconds=1)

    print(f"Fetching {start:%Y-%m}")

    while start < final:
        end = min(start + datetime.timedelta(days=7), final)

        try:
            wd = uo_pyfetch.get_sensor_data_by_name(
            'PER_EMLFLOOD_UO-WDENTONFS',
            start=start,
            end=end,
            variables=["Wind Direction", "Wind Speed"],
            limit=-1
            )
            west_denton.append(wd)

            birt = uo_pyfetch.get_sensor_data_by_name(
            'PER_EMLFLOOD_UO-BIRTLEYFS',
            start=start,
            end=end,
            variables=["Wind Direction", "Wind Speed"],
            limit=-1
            )
            birtley.append(birt)

        except Exception as e:
            print(f"Failed for {month}: {e}")
        start=end

westdenton_wind_2025 = pd.concat(west_denton, ignore_index=True)
birtley_wind_2025 = pd.concat(birtley, ignore_index=True)

westdenton_wind_2025.to_csv('2025-wind-UO_westdenton.csv')
birtley_wind_2025.to_csv('2025-wind-UO_birtley.csv')